# Full Feature Preparation Pipeline
Take your Week 4 ETL dataset (or any dataset with 8+ columns, mixed types, and some nulls). Apply the full prep pipeline: 

1. drop low-value columns with justification, 
1. impute missing values using at least 2 strategies, 
1. encode all categoricals appropriately, 
1. scale numerics with StandardScaler, 
1. perform a 80/20 train-test split. Output: a clean X_train, X_test, y_train, y_test and a written comment for every decision made

In [12]:
!pip install scikit-learn pandas pip

In [13]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
students_file_path = "../day_6/clean_students.csv"
df = pd.read_csv(students_file_path)

print("=" * 80)
print("STEP 1: INITIAL DATA EXPLORATION")
print("=" * 80)
print(f"\nDataset Shape: {df.shape}")
print(f"\nColumn Names and Types:\n{df.dtypes}")
print(f"\nFirst few rows:\n{df.head()}")
print(f"\nMissing Values:\n{df.isnull().sum()}")
print(f"\nBasic Statistics:\n{df.describe()}")

STEP 1: INITIAL DATA EXPLORATION

Dataset Shape: (35, 9)

Column Names and Types:
name                      str
math_score            float64
english_score         float64
science_score           int64
social_study_score    float64
math_grade                str
english_grade             str
science_grade             str
social_study_grade        str
dtype: object

First few rows:
       name  math_score  english_score  science_score  social_study_score  \
0     Alice        85.0           90.0             88                92.0   
1       Bob        92.0           78.0             85                40.0   
2   Charlie        48.0           95.0             50                42.0   
3     Grace        12.0           92.0             85                90.0   
4  Isabella        85.0           90.0             88                92.0   

  math_grade english_grade science_grade social_study_grade  
0          B             A             B                  A  
1          A             B    

## Step 2: Drop Low-Value Columns

**Decision:** Drop the 'name' column (identifier, non-predictive)

In [14]:
print("\n" + "=" * 80)
print("STEP 2: DROP LOW-VALUE COLUMNS")
print("=" * 80)

# Decision: Drop 'name' column
# Justification:
# - The 'name' column is an identifier with unique values for each student
# - Identifiers are not predictive features and should be removed before modeling
# - Including identifiers can lead to data leakage and overfitting
# - This is standard practice in ML pipelines

columns_to_drop = ['name']
df_cleaned = df.drop(columns=columns_to_drop)

print(f"\n✓ Dropped columns: {columns_to_drop}")
print(f"  Justification: Identifiers are non-predictive and can cause data leakage")
print(f"  New shape: {df_cleaned.shape}")
print(f"  Remaining columns: {df_cleaned.columns.tolist()}")


STEP 2: DROP LOW-VALUE COLUMNS

✓ Dropped columns: ['name']
  Justification: Identifiers are non-predictive and can cause data leakage
  New shape: (35, 8)
  Remaining columns: ['math_score', 'english_score', 'science_score', 'social_study_score', 'math_grade', 'english_grade', 'science_grade', 'social_study_grade']


## Step 3: Impute Missing Values

**Strategy 1:** Mean imputation for numeric score columns (math_score, english_score, science_score, social_study_score)
**Strategy 2:** Forward fill imputation for categorical grade columns (grade columns retain the score-based pattern)

In [15]:
print("\n" + "=" * 80)
print("STEP 3: IMPUTE MISSING VALUES")
print("=" * 80)

# Check for missing values first
missing_before = df_cleaned.isnull().sum()
print(f"\nMissing values before imputation:\n{missing_before}")

# Identify numeric and categorical columns
numeric_cols = df_cleaned.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_cleaned.select_dtypes(include=['object']).columns.tolist()

print(f"\nNumeric columns: {numeric_cols}")
print(f"Categorical columns: {categorical_cols}")

# Strategy 1: Mean Imputation for Numeric Columns
# Justification: Mean imputation preserves the distribution and is suitable for 
# score data where values are continuous and normally distributed
print("\n--- STRATEGY 1: Mean Imputation for Numeric Columns ---")
numeric_imputer = SimpleImputer(strategy='mean')
df_cleaned[numeric_cols] = numeric_imputer.fit_transform(df_cleaned[numeric_cols])
print(f"✓ Applied mean imputation to numeric columns: {numeric_cols}")
print(f"  Justification: Preserves distribution, suitable for continuous score data")

# Strategy 2: Forward Fill then Backward Fill for Categorical Columns
# Justification: Forward fill propagates grade values forward in the sequence,
# and backward fill fills any remaining nulls. This maintains categorical integrity
print("\n--- STRATEGY 2: Forward Fill (then Backward Fill) for Categorical Columns ---")
df_cleaned[categorical_cols] = df_cleaned[categorical_cols].ffill().bfill()
print(f"✓ Applied forward/backward fill to categorical columns: {categorical_cols}")
print(f"  Justification: Maintains categorical relationships and data integrity")

# Verify no missing values remain
print(f"\nMissing values after imputation:\n{df_cleaned.isnull().sum().sum()} total missing values")
print(f"\nDataset after imputation:\n{df_cleaned.head()}")


STEP 3: IMPUTE MISSING VALUES

Missing values before imputation:
math_score            0
english_score         0
science_score         0
social_study_score    0
math_grade            0
english_grade         0
science_grade         0
social_study_grade    0
dtype: int64

Numeric columns: ['math_score', 'english_score', 'science_score', 'social_study_score']
Categorical columns: ['math_grade', 'english_grade', 'science_grade', 'social_study_grade']

--- STRATEGY 1: Mean Imputation for Numeric Columns ---
✓ Applied mean imputation to numeric columns: ['math_score', 'english_score', 'science_score', 'social_study_score']
  Justification: Preserves distribution, suitable for continuous score data

--- STRATEGY 2: Forward Fill (then Backward Fill) for Categorical Columns ---
✓ Applied forward/backward fill to categorical columns: ['math_grade', 'english_grade', 'science_grade', 'social_study_grade']
  Justification: Maintains categorical relationships and data integrity

Missing values afte

## Step 4: Encode Categorical Variables

**Encoding Strategy:** Ordinal encoding for grade columns (F=0, C=1, B=2, A=3)

In [16]:
print("\n" + "=" * 80)
print("STEP 4: ENCODE CATEGORICAL VARIABLES")
print("=" * 80)

# Create ordinal mapping for grades
# Justification: Grades have an ordinal relationship (F < C < B < A)
# Ordinal encoding preserves this relationship better than one-hot encoding
grade_mapping = {'F': 0, 'C': 1, 'B': 2, 'A': 3}

print("\n--- ORDINAL ENCODING FOR GRADE COLUMNS ---")
print(f"Grade mapping: {grade_mapping}")
print("Justification: Grades are ordinal (F < C < B < A)")
print("             Ordinal encoding preserves this relationship")
print("             Reduces dimensionality vs. one-hot encoding\n")

# Apply ordinal encoding to all categorical (grade) columns
for col in categorical_cols:
    df_cleaned[col] = df_cleaned[col].map(grade_mapping)
    print(f"✓ Encoded {col}")

print(f"\nDataset after encoding:\n{df_cleaned.head()}")
print(f"\nData types after encoding:\n{df_cleaned.dtypes}")


STEP 4: ENCODE CATEGORICAL VARIABLES

--- ORDINAL ENCODING FOR GRADE COLUMNS ---
Grade mapping: {'F': 0, 'C': 1, 'B': 2, 'A': 3}
Justification: Grades are ordinal (F < C < B < A)
             Ordinal encoding preserves this relationship
             Reduces dimensionality vs. one-hot encoding

✓ Encoded math_grade
✓ Encoded english_grade
✓ Encoded science_grade
✓ Encoded social_study_grade

Dataset after encoding:
   math_score  english_score  science_score  social_study_score  math_grade  \
0        85.0           90.0           88.0                92.0           2   
1        92.0           78.0           85.0                40.0           3   
2        48.0           95.0           50.0                42.0           0   
3        12.0           92.0           85.0                90.0           0   
4        85.0           90.0           88.0                92.0           2   

   english_grade  science_grade  social_study_grade  
0              3              2                   3 

## Step 5: Feature Scaling with StandardScaler

**Scaling Method:** StandardScaler (zero-mean, unit-variance normalization)

In [17]:
print("\n" + "=" * 80)
print("STEP 5: FEATURE SCALING WITH STANDARDSCALER")
print("=" * 80)

# Prepare data for scaling - all features need to be scaled
X = df_cleaned.copy()

# Show statistics before scaling
print("\n--- BEFORE SCALING ---")
print(f"Feature statistics:\n{X.describe()}")

# Apply StandardScaler
# Justification: StandardScaler transforms features to have mean=0 and std=1
# This is crucial for distance-based algorithms (KNN, K-means, etc.)
# and helps with model convergence in neural networks and linear models
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("\n--- AFTER SCALING ---")
print(f"Feature statistics after StandardScaler:\n{X_scaled.describe()}")
print("\nJustification for StandardScaler:")
print("  - Transforms features to mean=0, std=1 (zero-mean, unit-variance)")
print("  - Prevents features with larger scales from dominating")
print("  - Improves convergence in gradient-based algorithms")
print("  - Required for many ML algorithms (KNN, SVM, Neural Networks)")
print(f"\nScaled data (first 5 rows):\n{X_scaled.head()}")


STEP 5: FEATURE SCALING WITH STANDARDSCALER

--- BEFORE SCALING ---
Feature statistics:
       math_score  english_score  science_score  social_study_score  \
count   35.000000       35.00000      35.000000           35.000000   
mean    62.571429       84.40000      73.228571           63.685714   
std     29.648218       16.09567      19.570515           24.603648   
min      0.000000        0.00000       0.000000            0.000000   
25%     44.500000       80.50000      60.500000           43.500000   
50%     72.000000       88.00000      80.000000           52.000000   
75%     86.500000       92.00000      87.000000           88.000000   
max    100.000000       98.00000      94.000000           95.000000   

       math_grade  english_grade  science_grade  social_study_grade  
count   35.000000      35.000000      35.000000           35.000000  
mean     1.371429       2.400000       1.628571            1.228571  
std      1.190297       0.650791       0.843163            1.

## Step 6: Train-Test Split (80/20)

**Target Variable:** social_study_grade (predicted based on other scores and grades)
**Features:** All other columns

In [18]:
print("\n" + "=" * 80)
print("STEP 6: TRAIN-TEST SPLIT (80/20)")
print("=" * 80)

# Define target variable
# Justification: Using social_study_grade as target to predict based on other academic metrics
target_col = 'social_study_grade'
print(f"\n✓ Target Variable: {target_col}")
print(f"  Justification: Predicting social studies grade based on other academic performance")

# Separate features (X) and target (y)
y = X_scaled[target_col]
X = X_scaled.drop(columns=[target_col])

print(f"\nFeatures (X): {X.columns.tolist()}")
print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Perform 80/20 train-test split
# Justification: 80/20 split is standard practice that balances model training with testing data
# train_size=0.8 means 80% for training, 20% for testing
# random_state=42 ensures reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\n--- TRAIN-TEST SPLIT RESULTS ---")
print(f"Training set size: {X_train.shape[0]} samples (80%)")
print(f"Testing set size: {X_test.shape[0]} samples (20%)")
print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

print(f"\nJustification for 80/20 split:")
print("  - 80% training data ensures sufficient samples for model learning")
print("  - 20% test data provides robust evaluation of model generalization")
print("  - Standard practice that balances training and evaluation")
print("  - random_state=42 ensures reproducible splits for debugging/comparison")


STEP 6: TRAIN-TEST SPLIT (80/20)

✓ Target Variable: social_study_grade
  Justification: Predicting social studies grade based on other academic performance

Features (X): ['math_score', 'english_score', 'science_score', 'social_study_score', 'math_grade', 'english_grade', 'science_grade']
Features shape: (35, 7)
Target shape: (35,)

--- TRAIN-TEST SPLIT RESULTS ---
Training set size: 28 samples (80%)
Testing set size: 7 samples (20%)

X_train shape: (28, 7)
X_test shape: (7, 7)
y_train shape: (28,)
y_test shape: (7,)

Justification for 80/20 split:
  - 80% training data ensures sufficient samples for model learning
  - 20% test data provides robust evaluation of model generalization
  - Standard practice that balances training and evaluation
  - random_state=42 ensures reproducible splits for debugging/comparison


## Step 7: Final Output - Ready for Modeling

In [19]:
print("\n" + "=" * 80)
print("STEP 7: FINAL OUTPUT SUMMARY")
print("=" * 80)

print("\n✓ FEATURE PREPARATION PIPELINE COMPLETE!\n")

print("=" * 80)
print("PREPARED DATASETS:")
print("=" * 80)
print(f"\nX_train (Training Features):")
print(f"  Shape: {X_train.shape}")
print(f"  Columns: {X_train.columns.tolist()}")
print(f"  First row:\n{X_train.iloc[0]}\n")

print(f"X_test (Testing Features):")
print(f"  Shape: {X_test.shape}")
print(f"  Columns: {X_test.columns.tolist()}\n")

print(f"y_train (Training Target):")
print(f"  Shape: {y_train.shape}")
print(f"  First 5 values: {y_train.head().values}\n")

print(f"y_test (Testing Target):")
print(f"  Shape: {y_test.shape}")
print(f"  First 5 values: {y_test.head().values}\n")

# Verification
print("=" * 80)
print("VERIFICATION:")
print("=" * 80)
print(f"✓ All values are numeric: {X_train.dtypes.nunique() == 1 and X_train.dtypes.unique()[0] == 'float64'}")
print(f"✓ No missing values in X_train: {X_train.isnull().sum().sum() == 0}")
print(f"✓ No missing values in X_test: {X_test.isnull().sum().sum() == 0}")
print(f"✓ No missing values in y_train: {y_train.isnull().sum() == 0}")
print(f"✓ No missing values in y_test: {y_test.isnull().sum() == 0}")
print(f"✓ Features are scaled (mean≈0): {abs(X_train.mean().mean()) < 0.01}")
print(f"✓ Features are scaled (std≈1): {abs(X_train.std().mean() - 1) < 0.1}")
print(f"✓ Train-test split ratio: {X_train.shape[0]/(X_train.shape[0]+X_test.shape[0]):.0%} train, {X_test.shape[0]/(X_train.shape[0]+X_test.shape[0]):.0%} test")

print("\n" + "=" * 80)
print("DECISION SUMMARY:")
print("=" * 80)
print("""
1. DROPPED COLUMNS: 'name'
   → Rationale: Identifiers are non-predictive and can cause data leakage
   
2. MISSING VALUE IMPUTATION:
   → Strategy 1 (Numeric): Mean imputation for score columns
     Rationale: Preserves distribution, suitable for continuous data
   → Strategy 2 (Categorical): Forward/Backward fill for grade columns
     Rationale: Maintains categorical relationships and integrity
     
3. CATEGORICAL ENCODING: Ordinal encoding for grades (F=0, C=1, B=2, A=3)
   → Rationale: Grades are ordinal, preserves ranking relationship
   
4. NUMERIC SCALING: StandardScaler (zero-mean, unit-variance)
   → Rationale: Prevents large-scale features from dominating
             Required for distance-based and gradient algorithms
             
5. TARGET VARIABLE: social_study_grade
   → Rationale: Predicting social studies performance based on other metrics
   
6. TRAIN-TEST SPLIT: 80/20 with random_state=42
   → Rationale: Sufficient training data while ensuring robust testing
             Reproducible results for debugging and comparison
""")


STEP 7: FINAL OUTPUT SUMMARY

✓ FEATURE PREPARATION PIPELINE COMPLETE!

PREPARED DATASETS:

X_train (Training Features):
  Shape: (28, 7)
  Columns: ['math_score', 'english_score', 'science_score', 'social_study_score', 'math_grade', 'english_grade', 'science_grade']
  First row:
math_score            0.322658
english_score        -0.592534
science_score        -0.167380
social_study_score   -0.729319
math_grade           -0.316603
english_grade        -0.623610
science_grade        -0.756376
Name: 12, dtype: float64

X_test (Testing Features):
  Shape: (7, 7)
  Columns: ['math_score', 'english_score', 'science_score', 'social_study_score', 'math_grade', 'english_grade', 'science_grade']

y_train (Training Target):
  Shape: (28,)
  First 5 values: [-1.02613901 -0.19090958  0.64431985  1.47954928  1.47954928]

y_test (Testing Target):
  Shape: (7,)
  First 5 values: [-1.02613901  0.64431985 -1.02613901 -1.02613901 -1.02613901]

VERIFICATION:
✓ All values are numeric: True
✓ No missing 